This 3-part notebook is intended to streamline adding raster or vector data covering the Stream Classification project study area to the dataset used to classify streams. If run in "ADD_NEW_DATA" mode cell, 1 of 3, identifies the new data (checking to see if a new raster or a new vector is present and processes/adds that only) by adding it first to two HUC12-based staging polygon feature classes--one composed of HUC12 shapes for the whole study are and the other composed of the same number of polygons but consisting of combinations of dissolved HUC12s where every given HUC12 number from the other dataset consists of not only that HUC but of every upstream HUC from that one dissolved into a single shape. If it is run in "RECALCULATE_FOR_HUC12" mode, it will recalculate all raster and vector metrics for the HUC12 number/s identified in the list recalc_huc12s. Need to configure parameters in cell one, but then cell 2 can be run with no changes if output of cell 1 looks good. Cell three will do a "dry run" to show you what it's about to do and will require a "yes" to move forward with overwriting or adding new fields (with update cursor) to Catchments_Final, the main NHDPlusV2 dataset with all basin characteristics.  

In [ ]:
# CELL 1: configuration & discovery
import arcpy, os, re
from datetime import datetime

def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"[{now()}] === CELL 1 starting ===")

# ---------------- USER EDITABLE INPUTS ----------------
gdb = r"C:\StreamClassProject\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb"
catchments_fc     = os.path.join(gdb, "Catchments_Final")                    # master table
local_huc_fc      = os.path.join(gdb, "ZonalChFeature_HUC12_EntireStudyArea_cleaned")
upstream_huc_fc   = os.path.join(gdb, "Catchments_All_Upstream")

# folder holding rasters (on disk)
rasters_folder    = r"C:\StreamClassRasters_5070"

# vector gdb containing source polygons for area-weighted fields
vectors_gdb       = r"C:\StreamClassProject\StreamClassProcessing\VectorData.gdb"

# HUC field (choose string HUC field as discussed)
huc_id_field = "HUC12"    # <-- using the STRING HUC field (per our last decision)

# comid and is_mainstem fields in Catchments_Final (used in CELL 3)
comid_field = "comid"
is_mainstem_field = "IsMainstem"  # 1 == upstream/mainstem values should be used, 0 == local values

# Run mode: one of these strings:
# - "ADD_NEW_DATA"            -> compute only rasters/vectors missing in local/upstream; create fields in Catchments_Final if needed (by running cell 3)
# - "RECALCULATE_FOR_HUC12"   -> recompute existing raster & vector fields but only for HUCs in recalc_huc12s

run_mode = "RECALCULATE_FOR_HUC12"   # change as needed

# list of HUC12 identifiers (strings OR ints OK). Used only for RECALCULATE_FOR_HUC12
recalc_huc12s = ["131211100903"]   # can supply many HUC12 values, i.e., a comma-seperated list

# ---------------- RASTER LIST (explicit filenames with .tif) ----------------
# (Cell 2 expects raster_files variable)
raster_files = [
    "Aridity_daymetws.tif",
    "DepthToBedrockMean_USGS.tif",
    "ElevMedian_daymetws.tif",
    "InterAnnPrecipCV_daymetws.tif",
    "PercPrecipSnow_daymetws.tif",
    "PotEvapotransEra5_daymetws.tif",
    "PrecDaysMean_daymetws.tif",
    "PrecipIntensity_daymetws.tif",
    "PrecipSeasInd_daymetws.tif",
    "PrecMean_daymetws.tif",
    "SimpPrecIntensity_daymetws.tif",
    "SoilWaterContent_WCPF2Mean.tif",
    "SRadMean_daymetws.tif",
    "SWE_April_daymetws.tif",
    "TempJulyMean_daymetws.tif",
    "TempMax_daymetws.tif",
    "TempMean_daymetws.tif",
    "TempMin_daymetws.tif",
    "WinterSumPrecipRatio_daymetws.tif"
]

# ---------------- STAT MAP (explicit mapping: base raster name -> statistic) ----------------
# Use explicit pairing (you requested this). Base names match raster_files minus .tif
STAT_MAP = {
    "Aridity_daymetws": "MEAN",
    "DepthToBedrockMean_USGS": "MEAN",
    "ElevMedian_daymetws": "MEDIAN",
    "InterAnnPrecipCV_daymetws": "MEAN",
    "PercPrecipSnow_daymetws": "MEAN",
    "PotEvapotransEra5_daymetws": "MEAN",
    "PrecDaysMean_daymetws": "MEAN",
    "PrecipIntensity_daymetws": "MEAN",
    "PrecipSeasInd_daymetws": "MEAN",
    "PrecMean_daymetws": "MEAN",
    "SimpPrecIntensity_daymetws": "MEAN",
    "SoilWaterContent_WCPF2Mean": "MEAN",
    "SRadMean_daymetws": "MEAN",
    "SWE_April_daymetws": "MEAN",
    "TempJulyMean_daymetws": "MEAN",
    "TempMax_daymetws": "MEAN",
    "TempMean_daymetws": "MEAN",
    "TempMin_daymetws": "MEAN",
    "WinterSumPrecipRatio_daymetws": "MEAN"
}

# ---------------- VECTOR MAP (Option A: explicit mapping in vectors_gdb) ----------------
# Keys are feature class names inside vectors_gdb; values are lists of field names inside that fc
# The script will produce area-weighted means of these fields into the HUC feature classes.
VECTOR_MAP = {
    "HUC12_StudyArea_CaravanAttributes_Albers": ["aet_mm_syr", "ari_ix_sav", "cly_pc_sav", "cmi_ix_syr", "slt_pc_sav", "snd_pc_sav", "run_mm_syr"],
    "GLHYMPS_StudyArea": ["Porosity", "Permeability_permafrost"]
}

# ---------------- small helpers ----------------
def sanitize_field_name(name):
    return re.sub(r'[^A-Za-z0-9_]', '_', name)[:60]

def raster_fullpath(name):
    # Strip any existing .tif or .tif.tif situations, case-insensitive
    base = re.sub(r'\.tif$', '', name, flags=re.IGNORECASE)
    return os.path.join(rasters_folder, base + ".tif")


# ---------------- environment ----------------
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

# ---------------- validations & discovery ----------------
print(f"[{now()}] Validating paths & inputs...")
for p,label in [(gdb,"gdb"), (rasters_folder,"rasters_folder"), (vectors_gdb,"vectors_gdb")]:
    exists = os.path.exists(p) or arcpy.Exists(p)
    print(f"  {label}: {p} | Exists? {exists}")

# required feature classes
for fc in (catchments_fc, local_huc_fc, upstream_huc_fc):
    exists = arcpy.Exists(fc)
    print(f"  Checking {fc} -> Exists? {exists}")
    if not exists:
        raise RuntimeError(f"Required feature class not found: {fc}")

# validate raster files exist on disk (compare using raster_files list)
missing_rasters = []
for r in raster_files:
    if not os.path.exists(raster_fullpath(r)):
        missing_rasters.append(r)
if missing_rasters:
    print(f"[{now()}] WARNING: missing raster files in rasters_folder: {missing_rasters}")
else:
    print(f"[{now()}] All expected rasters present in rasters_folder ({len(raster_files)}).")

# validate VECTOR_MAP feature classes + fields inside vectors_gdb
arcpy.env.workspace = vectors_gdb
vec_fcs = arcpy.ListFeatureClasses() or []
vec_missing = []
vec_field_issues = {}
for vfc, fields in VECTOR_MAP.items():
    if vfc not in vec_fcs:
        vec_missing.append(vfc)
    else:
        fns = [f.name for f in arcpy.ListFields(os.path.join(vectors_gdb, vfc))]
        missing_fields = [f for f in fields if f not in fns]
        if missing_fields:
            vec_field_issues[vfc] = missing_fields

if vec_missing:
    raise RuntimeError("Missing vector feature class(es) in vectors_gdb: " + ", ".join(vec_missing))
if vec_field_issues:
    raise RuntimeError("Missing fields in vector FCs: " + str(vec_field_issues))

# inspect HUC fields presence (local / upstream / catchments)
local_fields = [f.name for f in arcpy.ListFields(local_huc_fc)]
up_fields = [f.name for f in arcpy.ListFields(upstream_huc_fc)]
nhd_fields = [f.name for f in arcpy.ListFields(catchments_fc)]

print(f"[{now()}] Local fields count: {len(local_fields)}, Upstream: {len(up_fields)}, Catchments_Final: {len(nhd_fields)}")

# summarize which raster-derived fields are present in local/upstream
raster_fieldnames = [os.path.splitext(r)[0] for r in raster_files]
raster_presence_local = [r for r in raster_fieldnames if r in local_fields]
raster_presence_up = [r for r in raster_fieldnames if r in up_fields]

# vector fields presence
vector_fieldnames = []
for vfc, flds in VECTOR_MAP.items():
    vector_fieldnames += flds
vector_presence_local = [v for v in vector_fieldnames if v in local_fields]
vector_presence_up = [v for v in vector_fieldnames if v in up_fields]

# ---------------- Job summary printout ----------------
print("\n=== JOB SUMMARY ===")
print("Run mode:", run_mode)
print("Recalc HUC12s (if applicable):", recalc_huc12s)
print("Raster count (declared):", len(raster_files))
print("Raster STAT_MAP explicit keys:", len(STAT_MAP))
print("Raster fields already present in local:", len(raster_presence_local))
print("Raster fields already present in upstream:", len(raster_presence_up))
print("Vector sources (VECTOR_MAP keys):", list(VECTOR_MAP.keys()))
print("Requested vector fields:", vector_fieldnames)
print("Vector present in local:", vector_presence_local)
print("Vector present in upstream:", vector_presence_up)

# === EXPORTS FOR CELL 2 ===
# (Cell 2 depends on these names)

local_fc = local_huc_fc
up_fc    = upstream_huc_fc
rasters  = raster_files

# Mode flags expected by Cell 2
mode = run_mode
ADD_NEW_DATA = (run_mode == "ADD_NEW_DATA")

# ------------------------------------------------------------
# Authoritative list of all metric fields written by Cell 2
# ------------------------------------------------------------

# Raster-derived metrics (field names already created by Cell 2)
raster_metric_fields = [r.replace(".tif", "") for r in raster_files]

# Vector-derived metrics (area-weighted means written by Cell 2)
vector_metric_fields = []
for v in VECTOR_MAP.values():
    if isinstance(v, list):
        vector_metric_fields.extend(v)
    else:
        vector_metric_fields.extend(v["fields"])

combined_metric_fields = sorted(set(raster_metric_fields + vector_metric_fields))

print(f"Combined metric fields ({len(combined_metric_fields)}):")
for f in combined_metric_fields:
    print("  ", f)

print("\nCell 1 complete. Inspect output above. Run CELL 2 next.")


[2025-12-26 09:49:40] === CELL 1 starting ===
[2025-12-26 09:49:40] Validating paths & inputs...
  gdb: C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb | Exists? True
  rasters_folder: C:\StreamClassRasters_5070 | Exists? True
  vectors_gdb: C:\Users\CN0401\Documents\ArcGIS\Projects\StreamClassProcessing\VectorData.gdb | Exists? True
  Checking C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\Catchments_Final -> Exists? True
  Checking C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\ZonalChFeature_HUC12_EntireStudyArea_cleaned -> Exists? True
  Checking C:\Users\CN0401\Documents\ArcGIS\Projects\Clipping Big Rasters\StreamClassCommons_REPAIRED.gdb\Catchments_All_Upstream -> Exists? True
[2025-12-26 09:49:40] All expected rasters present in rasters_folder (19).
[2025-12-26 09:49:40] Local fields count: 35, Upstream: 37, Catchments_Final: 152

=== JOB

Module 2 of 3: Calculations done here and written to local and upstream-dissolved HUC12 feature classes that can be checked before module 3 where they are added to the master/nhdplusv2 catchment feature ("Catchments_Final"). If in the "ADD_NEW_DATA" mode, new fields are created and values added via update cursor. If run in recalculate for HUC12 mode, only comid lines in the HUCs that are identified as needing recalculation in cell 1 are updated/recalculated in cell 2 -- in this case, no new fields are created in Catchments_Final by cell 3. The existing fields are overwritten with what has been recalculated in the local and upstream HUC12 tables. 

In [ ]:
# ================================================================
# CELL 2 — Compute Zonal Stats (Raster + Vector) for LOCAL + UPSTREAM
# ================================================================

import arcpy
import os
from datetime import datetime
import re
from collections import defaultdict
import traceback

# ------------------------------------------------
# Helper logging function
# ------------------------------------------------
def log(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

def warn(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] [WARNING] {msg}")

log("=== CELL 2 starting ===")

# --------------------------------------------------------------------
# CHECK OUT SPATIAL ANALYST EXTENSION
# --------------------------------------------------------------------
try:
    sa_status = arcpy.CheckExtension("Spatial")
    if sa_status != "Available":
        raise RuntimeError(f"Spatial Analyst not available (status={sa_status})")

    checkout_status = arcpy.CheckOutExtension("Spatial")
    if checkout_status != "CheckedOut":
        raise RuntimeError(f"Failed to check out Spatial Analyst (status={checkout_status})")

    log("[OK] Spatial Analyst extension checked out.")
except Exception as e:
    raise RuntimeError(f"[FATAL] Could not check out Spatial Analyst: {e}")

# --------------------------------------------------------------------
# HELPER FOR PROPER SQL QUERY IN RECALCULATE_FOR_HUC12 mode
# --------------------------------------------------------------------

def build_in_clause(fc, field_name, values):
    """
    Safely build an IN (...) SQL clause for text or numeric fields.
    """
    if not values:
        return None

    field = arcpy.ListFields(fc, field_name)[0]
    delim = arcpy.AddFieldDelimiters(fc, field_name)

    if field.type == "String":
        quoted = ",".join(f"'{v}'" for v in values)
    else:
        quoted = ",".join(str(v) for v in values)

    return f"{delim} IN ({quoted})"

# --------------------------------------------------------------------
# HELPER FOR determining if geometry = same in vector field. If so, no need to intersect to add data to local HUC12 shape > for example, with Caravan data
# --------------------------------------------------------------------

def same_geometry(fc1, fc2):
    d1 = arcpy.Describe(fc1)
    d2 = arcpy.Describe(fc2)
    return (
        d1.shapeType == d2.shapeType and
        d1.spatialReference.factoryCode == d2.spatialReference.factoryCode and
        int(arcpy.management.GetCount(fc1)[0]) ==
        int(arcpy.management.GetCount(fc2)[0])
    )


# --------------------------------------------------------------------
# Helper: normalize raster filepath
# accepts entries in raster_files with or without ".tif"
# --------------------------------------------------------------------
def raster_fullpath(name):
    base = re.sub(r'\.tif$', '', name, flags=re.IGNORECASE)
    return os.path.join(rasters_folder, base + ".tif")

arcpy.env.overwriteOutput = True

# ------------------------------------------------
# Validate that Cell 1 variables exist (names that your Cell 1 provides)
# ------------------------------------------------
required = [
    "gdb",
    "rasters_folder",
    "local_huc_fc",
    "upstream_huc_fc",
    "catchments_fc",
    "raster_files",           # list of raster filenames (with .tif)
    "STAT_MAP",               # explicit raster -> statistic mapping (base name -> "MEAN"/"MEDIAN")
    "VECTOR_MAP",             # vector mapping from Cell 1
    "run_mode",               # run mode string from Cell 1
    "huc_id_field",           # the HUC text field name to use (e.g., "HUC12")
    "scratch_gdb",            # scratch gdb from Cell 1 / arcpy.env.scratchGDB
    "vectors_gdb",            # vector gdb path
    "recalc_huc12s"          # list (may be empty) used by RECALCULATE_FOR_HUC12
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(f"Missing required names from Cell 1: {missing}")

# Local variables mapped from Cell 1 names (keep these names in logs)
local_fc = local_huc_fc
up_fc = upstream_huc_fc
master_fc = catchments_fc
rasters = raster_files        # internal name for loop

log(f"Local HUC source: {local_fc}")
log(f"Upstream HUC source: {up_fc}")
log(f"Run mode: {run_mode}")

# =======================================================================
#  FUNCTION: choose_zone_field_for_fc(fc_path)
#  - Accepts case-insensitive match to huc_id_field
#  - Accepts any pre-existing text HUC field matching that name (case-insensitive)
#  - Otherwise creates temp <huc_id_field>_ztemp (lowercase) and populates it
# =======================================================================
def choose_zone_field_for_fc(fc_path):
    fields = arcpy.ListFields(fc_path)
    # 1 — case-insensitive direct match
    for f in fields:
        if f.name.upper() == huc_id_field.upper():
            log(f"Matched HUC field (case-insensitive): {f.name} in {os.path.basename(fc_path)}")
            return f.name, False
    # 2 — accept existing string field with matching name (redundant but explicit)
    for f in fields:
        if f.type == "String" and f.name.upper() == huc_id_field.upper():
            log(f"Matched TEXT HUC field: {f.name} in {os.path.basename(fc_path)}")
            return f.name, False
    # 3 — create temp text field and fill from numeric HUC field if available
    temp_field = f"{huc_id_field.lower()}_ztemp"
    log(f"No usable HUC text field in {os.path.basename(fc_path)} → creating temp: {temp_field}")
    if temp_field not in [f.name for f in fields]:
        # add text field and fill from the declared huc_id_field (if exists)
        arcpy.management.AddField(fc_path, temp_field, "TEXT", field_length=50)
        # try to fill it from whatever field that matches the declared HUC (case-insensitive)
        # prefer numeric HUC field (the one we discussed previously)
        # find candidate source field (case-insensitive match to huc_id_field)
        src = None
        for f in fields:
            if f.name.upper() == huc_id_field.upper():
                src = f.name
                break
        if src is None:
            # fall back to first string or numeric field
            for f in fields:
                if f.type in ("String", "Integer", "SmallInteger", "Double", "BigInteger"):
                    src = f.name
                    break
            if src is None:
                warn(f"Could not find any reasonable source to populate {temp_field} in {os.path.basename(fc_path)}")
        if src:
            with arcpy.da.UpdateCursor(fc_path, [src, temp_field]) as cur:
                for row in cur:
                    row[1] = str(row[0]) if row[0] is not None else None
                    cur.updateRow(row)
    return temp_field, True

# ----------------------------------------------------------------------
# Determine zone/text fields for both local and upstream feature classes
# ----------------------------------------------------------------------
zone_local_field, local_tmp_created = choose_zone_field_for_fc(local_fc)
zone_up_field, up_tmp_created = choose_zone_field_for_fc(up_fc)

log(f"Using zone field for LOCAL: '{zone_local_field}' (tmp_created={local_tmp_created})")
log(f"Using zone field for UPSTREAM: '{zone_up_field}' (tmp_created={up_tmp_created})")

# =======================================================================
# Ensure in-memory feature layers for ZonalStatisticsAsTable (it often requires layers)
# =======================================================================
try:
    if arcpy.Exists("local_layer"):
        try: arcpy.management.Delete("local_layer")
        except: pass
    arcpy.management.MakeFeatureLayer(local_fc, "local_layer")
    local_layer = "local_layer"
    log("Created feature layer: local_layer")
except Exception:
    raise RuntimeError("ERROR: Could not create feature layer local_layer")

try:
    if arcpy.Exists("up_layer"):
        try: arcpy.management.Delete("up_layer")
        except: pass
    arcpy.management.MakeFeatureLayer(up_fc, "up_layer")
    up_layer = "up_layer"
    log("Created feature layer: up_layer")
except Exception:
    raise RuntimeError("ERROR: Could not create feature layer up_layer")

# =======================================================================
# detect_zone_field: figure out which field the ZonalStatisticsAsTable output used
# =======================================================================
def detect_zone_field(table_path, candidate_zone_name):
    fields = arcpy.ListFields(table_path)
    names = [f.name for f in fields]
    # 1 — exact / case-insensitive match
    for n in names:
        if n.upper() == candidate_zone_name.upper():
            return n
    # 2 — common fallback names
    for fallback in ("ZONE", "VALUE", "ZONE_CODE"):
        if fallback in names:
            return fallback
    # 3 — first usable field
    for f in fields:
        if f.type in ("String", "Integer", "SmallInteger", "OID"):
            return f.name
    warn(f"No usable zone field found in zonal output {table_path}; candidate was {candidate_zone_name}")
    raise RuntimeError(f"No usable zone field found in {table_path}")

# =======================================================================
# RASTER PROCESSING
# =======================================================================
log("=== Processing rasters ===")

for rfile in rasters:
    base = os.path.splitext(rfile)[0]   # base name (no .tif)
    log(f"[RASTER] {rfile}")

    tif_path = raster_fullpath(rfile)
    if not os.path.exists(tif_path):
        warn(f"Raster file missing (skipping): {tif_path}")
        continue

    # look up statistic by base name
    stat_type = STAT_MAP.get(base)
    if stat_type is None:
        warn(f"No statistic defined for raster '{base}' in STAT_MAP — skipping")
        continue

    out_field = base  # field name = base (never add prefixes/suffixes)

    # determine if we must compute for local/up based on run_mode
    local_fields = [f.name for f in arcpy.ListFields(local_fc)]
    up_fields = [f.name for f in arcpy.ListFields(up_fc)]

    compute_local = compute_up = False
    if run_mode == "ADD_NEW_DATA":
        compute_local = out_field not in local_fields
        compute_up = out_field not in up_fields
    elif run_mode == "RECALC_RASTER_ONLY":
        compute_local = compute_up = True
    elif run_mode == "RECALCULATE_FOR_HUC12":
        compute_local = compute_up = True

    if not compute_local and not compute_up:
        log(f"⏩ Skipping raster {base} (already present where needed)")
        continue

    # attempt to project raster to scratch GDB if helper exists; otherwise use tif_path
    try:
        raster_for_use = ensure_projected_raster_to_scratch(tif_path)
    except Exception:
        raster_for_use = tif_path

    # --- LOCAL zonal ---
    if compute_local:
        tbl_local = os.path.join("in_memory", f"zs_loc_{base}")
        try:
            if arcpy.Exists(tbl_local):
                arcpy.management.Delete(tbl_local)
            log(f"ZonalStatisticsAsTable (LOCAL) for {base} (stat={stat_type}) using zone '{zone_local_field}'")
            arcpy.sa.ZonalStatisticsAsTable(local_layer, zone_local_field, raster_for_use, tbl_local, "DATA", stat_type)

            # detect zone field used in output table
            try:
                zone_field_out = detect_zone_field(tbl_local, zone_local_field)
            except Exception as e:
                warn(f"Could not detect zone field in local zonal table for {base}: {e}")
                raise

            # read zone->value
            stats = {}
            if arcpy.Exists(tbl_local):
                with arcpy.da.SearchCursor(tbl_local, [zone_field_out, stat_type]) as scur:
                    for z, v in scur:
                        stats[str(z)] = v

            # add out_field to local_fc if needed
            if out_field not in [f.name for f in arcpy.ListFields(local_fc)]:
                arcpy.management.AddField(local_fc, out_field, "DOUBLE")

            # update rows (respect RECALCULATE_FOR_HUC12 if used)
            only_hucs = recalc_huc12s if run_mode == "RECALCULATE_FOR_HUC12" else None
            where_clause = build_in_clause(local_fc, huc_id_field, only_hucs) if only_hucs else None

            with arcpy.da.UpdateCursor(local_fc, [zone_local_field, out_field], where_clause) as uc:
                for row in uc:
                    hid = str(row[0])
                    if hid in stats and stats[hid] is not None:
                        row[1] = float(stats[hid])
                    else:
                        if only_hucs:
                            # do not clobber other HUCs (we only iterate subset)
                            pass
                        else:
                            row[1] = None
                    uc.updateRow(row)

            # cleanup
            if arcpy.Exists(tbl_local):
                arcpy.management.Delete(tbl_local)
            log(f"✅ Local HUC field populated: {out_field}")

        except Exception:
            warn(f"ERROR computing LOCAL zonal for {base}")
            traceback.print_exc()
            try:
                if arcpy.Exists(tbl_local):
                    arcpy.management.Delete(tbl_local)
            except:
                pass

    # --- UPSTREAM zonal ---
    if compute_up:
        tbl_up = os.path.join("in_memory", f"zs_up_{base}")
        try:
            if arcpy.Exists(tbl_up):
                arcpy.management.Delete(tbl_up)
            log(f"ZonalStatisticsAsTable (UPSTREAM) for {base} (stat={stat_type}) using zone '{zone_up_field}'")
            arcpy.sa.ZonalStatisticsAsTable(up_layer, zone_up_field, raster_for_use, tbl_up, "DATA", stat_type)

            try:
                zone_field_out_up = detect_zone_field(tbl_up, zone_up_field)
            except Exception as e:
                warn(f"Could not detect zone field in upstream zonal table for {base}: {e}")
                raise

            stats_up = {}
            if arcpy.Exists(tbl_up):
                with arcpy.da.SearchCursor(tbl_up, [zone_field_out_up, stat_type]) as scur:
                    for z, v in scur:
                        stats_up[str(z)] = v

            if out_field not in [f.name for f in arcpy.ListFields(up_fc)]:
                arcpy.management.AddField(up_fc, out_field, "DOUBLE")

            only_hucs = recalc_huc12s if run_mode == "RECALCULATE_FOR_HUC12" else None
            where_clause = build_in_clause(up_fc, huc_id_field, only_hucs) if only_hucs else None

            with arcpy.da.UpdateCursor(up_fc, [zone_up_field, out_field], where_clause) as uc2:
                for row in uc2:
                    hid = str(row[0])
                    if hid in stats_up and stats_up[hid] is not None:
                        row[1] = float(stats_up[hid])
                    else:
                        if only_hucs:
                            pass
                        else:
                            row[1] = None
                    uc2.updateRow(row)

            if arcpy.Exists(tbl_up):
                arcpy.management.Delete(tbl_up)
            log(f"✅ Upstream HUC field populated: {out_field}")

        except Exception:
            warn(f"ERROR computing UPSTREAM zonal for {base}")
            traceback.print_exc()
            try:
                if arcpy.Exists(tbl_up):
                    arcpy.management.Delete(tbl_up)
            except:
                pass

log("=== Raster processing complete ===")

# =======================================================================
# VECTOR PROCESSING (area-weighted means)
# =======================================================================
log("=== Processing vectors (area-weighted means) ===")

for vname, vinfo in VECTOR_MAP.items():
    # support both Option A (list) and Option B (dict)
    if isinstance(vinfo, list):
        v_fc = os.path.join(vectors_gdb, vname)
        join_field = None
        v_fields = vinfo
    elif isinstance(vinfo, dict):
        v_fc = vinfo.get("fc_path") or os.path.join(vectors_gdb, vname)
        join_field = vinfo.get("join_field")
        v_fields = vinfo.get("fields") or []
    else:
        warn(f"Unsupported VECTOR_MAP entry for {vname}: {type(vinfo)}")
        continue

    if not arcpy.Exists(v_fc):
        warn(f"Vector FC not found: {v_fc}; skipping {vname}")
        continue

    log(f"[VECTOR] {vname} -> {v_fc} (fields: {v_fields})")

    zone_v_field, _ = choose_zone_field_for_fc(v_fc)

    for label, target_fc, zone_field in [("LOCAL", local_fc, zone_local_field), ("UPSTREAM", up_fc, zone_up_field)]:
        # -------------------------------------------------
        # IDENTICAL GEOMETRY SHORT-CIRCUIT (LOCAL only)
        #   Cursor-based attribute transfer (NO JOINS)
        # -------------------------------------------------
        if label == "LOCAL" and same_geometry(target_fc, v_fc):
            log(f"  {label}: geometry identical → cursor-based transfer (no join, no intersect)")

            target_fields = [f.name for f in arcpy.ListFields(target_fc)]
            needed = [f for f in v_fields if f not in target_fields]

            if not needed and run_mode != "RECALCULATE_FOR_HUC12":
                log(f"  {label}: all fields already present → nothing to do")
                continue

            log(f"  {label}: transferring fields {needed} via cursor")

            # build lookup from source vector (v_fc)
            lookup = {}
            with arcpy.da.SearchCursor(v_fc, [zone_v_field] + needed) as sc:
                for row in sc:
                    lookup[str(row[0])] = row[1:]

            # ensure fields exist on target_fc
            existing = [f.name for f in arcpy.ListFields(target_fc)]
            for fld in needed:
                if fld not in existing:
                    arcpy.management.AddField(target_fc, fld, "DOUBLE")

            # write values directly to target_fc
            only_hucs = recalc_huc12s if run_mode == "RECALCULATE_FOR_HUC12" else None
            where_clause = build_in_clause(target_fc, zone_field, only_hucs) if only_hucs else None

            with arcpy.da.UpdateCursor(target_fc, [zone_field] + needed, where_clause) as uc:
                for row in uc:
                    hid = str(row[0])
                    if hid in lookup:
                        for i, val in enumerate(lookup[hid], start=1):
                            row[i] = val
                        uc.updateRow(row)

            continue   # IMPORTANT: skip intersect path


        # ---------------------------------------------
        # 2. INTERSECT PATH (non-identical geometry)
        # ---------------------------------------------

        target_fields = [f.name for f in arcpy.ListFields(target_fc)]
        needed = [f for f in v_fields if f not in target_fields]

        if not needed:
            if run_mode == "RECALCULATE_FOR_HUC12":
                log(f"  {label}: fields already present → skipping field creation (values WILL be recomputed)")
            else:
                log(f"  {label}: all fields already present → skipping field creation")
                continue

        log(f"  {label}: will compute AW mean for fields: {needed}")

        stamp = datetime.now().strftime("%H%M%S")
        int_out = os.path.join(scratch_gdb, f"int_{vname}_{label}_{stamp}")

        try:
            if arcpy.Exists(int_out):
                arcpy.management.Delete(int_out)
        except:
            pass

        log(f"  {label}: geometry differs → intersect + AW mean")

        try:
            arcpy.analysis.Intersect(
                [target_fc, v_fc],
                int_out,
                join_attributes="ONLY_FID",
                output_type="INPUT"
            )

            # -------------------------------------------------
            # STEP 1: Join target HUC field INTO intersect output
            # -------------------------------------------------

            target_oid = arcpy.Describe(target_fc).OIDFieldName
            target_huc = zone_field

            log(f"    Joining HUC field '{target_huc}' from target FC into intersect output")

            arcpy.management.JoinField(
                int_out,
                f"FID_{os.path.basename(target_fc)}",
                target_fc,
                target_oid,
                [target_huc]
            )

            # now resolve the joined field name safely
            def find_field_ci(fc, desired):
                for f in arcpy.ListFields(fc):
                    if f.name.lower() == desired.lower():
                        return f.name
                raise RuntimeError(
                    f"Field '{desired}' not found (case-insensitive) in {fc}"
                )

            int_huc_field = find_field_ci(int_out, target_huc)
            log(f"    Using HUC field in intersect output: {int_huc_field}")

            # -------------------------------------------------
            # Existing join (unchanged)
            # -------------------------------------------------
            arcpy.management.JoinField(
                int_out,
                f"FID_{os.path.basename(v_fc)}",
                v_fc,
                arcpy.Describe(v_fc).OIDFieldName,
                v_fields
            )

            log(f"    Intersect fields: {[f.name for f in arcpy.ListFields(int_out)]}")

        except Exception:
            warn(f"  ERROR intersecting {target_fc} with {v_fc}")
            traceback.print_exc()
            continue


        # area field
        if "INT_AREA" not in [f.name for f in arcpy.ListFields(int_out)]:
            arcpy.management.AddField(int_out, "INT_AREA", "DOUBLE")
        arcpy.management.CalculateGeometryAttributes(int_out, [["INT_AREA", "AREA"]], area_unit="SQUARE_METERS")

        sums = defaultdict(lambda: [0.0, 0.0])
        search_fields = [int_huc_field, "INT_AREA"] + v_fields
        with arcpy.da.SearchCursor(int_out, search_fields) as sc:
            for rec in sc:
                hid = str(rec[0])
                area = float(rec[1] or 0.0)
                for idx, src in enumerate(v_fields, start=2):
                    val = rec[idx]
                    if val is None:
                        continue
                    try:
                        fval = float(val)
                    except:
                        continue
                    key = (hid, src)
                    sums[key][0] += fval * area
                    sums[key][1] += area

        # add & write fields
        for src_field in v_fields:
            if src_field not in target_fields:
                arcpy.management.AddField(target_fc, src_field, "DOUBLE")
                log(f"    Added field {src_field} to {target_fc}")

            target_fields = [f.name for f in arcpy.ListFields(target_fc)]

            only_hucs = recalc_huc12s if run_mode == "RECALCULATE_FOR_HUC12" else None
            where_clause = build_in_clause(target_fc, zone_field, only_hucs) if only_hucs else None

            with arcpy.da.UpdateCursor(target_fc, [zone_field, src_field], where_clause) as ucur:
                for urow in ucur:
                    hid = str(urow[0])
                    key = (hid, src_field)
                    if key in sums and sums[key][1] > 0.0:
                        urow[1] = sums[key][0] / sums[key][1]
                    else:
                        if only_hucs:
                            urow[1] = None if hid in [str(x) for x in only_hucs] else urow[1]
                        else:
                            urow[1] = None
                    ucur.updateRow(urow)

        try:
            arcpy.management.Delete(int_out)
        except:
            pass

        log(f"  ✔ Completed vector area-weighted for {vname} -> {target_fc}")

# Return Spatial Analyst license
try:
    arcpy.CheckInExtension("Spatial")
    log("[OK] Spatial Analyst extension checked in.")
except:
    warn("Unable to check in Spatial Analyst (may not be checked out).")

log("=== CELL 2 complete ===")


In [ ]:
arcpy.ListFields(local_fc, "huc12")[0].type
arcpy.ListFields(up_fc, "HUC12")[0].type


Module 3 of 3: uses update cursor to add the new field/s and data values (per comid) to the Stream Class master dataset ("Catchments_Final"). If DRY_RUN = True, the script will not write but will give a preview of what it is about to do (like add new fields in ADD_NEW_DATA mode or what comids it's going to recalculate in RECALCULATE_FOR_HUC12 mode). Recommended to use this first. Then, when ready to write to Catchments_Final, change to DRY_RUN = Fale. 

In [4]:
# ------------------------------------------------------------------------------------
# CELL 3 — Push HUC-level metrics into Catchments_Final (cursor-based, NO JOINS)
# ------------------------------------------------------------------------------------
# Supported modes (EXACTLY matching Cell 1):
#   - "ADD_NEW_DATA"
#   - "RECALCULATE_FOR_HUC12"
#
# DRY_RUN = True  -> preview updates only
# DRY_RUN = False -> write updates
# ------------------------------------------------------------------------------------

import arcpy

print("\n=== CELL 3 START ===")

DRY_RUN = False  # <- set False to write

# Inputs from Cell 1
local_fc       = local_huc_fc
up_fc          = upstream_huc_fc
catch_fc       = catchments_fc
mode           = run_mode
target_hucs    = recalc_huc12s
is_mainstem_f  = is_mainstem_field

# Metric fields = authoritative list assembled in Cell 1
metric_fields = combined_metric_fields

VERBOSE_DRY_RUN = (mode == "RECALCULATE_FOR_HUC12")

# ------------------------------------------------------------
# Helper: find HUC field (case-insensitive)
# ------------------------------------------------------------
def find_huc_field(fc):
    for f in arcpy.ListFields(fc):
        if f.name.lower() == huc_id_field.lower():
            return f.name
    raise RuntimeError(f"HUC field not found in {fc}")

local_huc_field = find_huc_field(local_fc)
up_huc_field    = find_huc_field(up_fc)
catch_huc_field = find_huc_field(catch_fc)

# ------------------------------------------------------------
# Read HUC-level values into dictionaries
# ------------------------------------------------------------
def read_huc_values(fc, huc_field):
    vals = {}
    with arcpy.da.SearchCursor(fc, [huc_field] + metric_fields) as cur:
        for row in cur:
            h = str(row[0])
            vals[h] = dict(zip(metric_fields, row[1:]))
    return vals

local_vals = read_huc_values(local_fc, local_huc_field)
up_vals    = read_huc_values(up_fc, up_huc_field)

print(f"Loaded HUC values: local={len(local_vals)}, upstream={len(up_vals)}")

# ------------------------------------------------------------
# Determine which fields to update
# ------------------------------------------------------------
catch_fields = [f.name for f in arcpy.ListFields(catch_fc)]

if mode == "ADD_NEW_DATA":
    fields_to_update = [f for f in metric_fields if f not in catch_fields]
elif mode == "RECALCULATE_FOR_HUC12":
    fields_to_update = [f for f in metric_fields if f in catch_fields]
else:
    raise RuntimeError(f"Unsupported mode: {mode}")

print(f"Fields to update: {fields_to_update}")

# ------------------------------------------------------------
# ADD FIELD CREATION (ADD_NEW_DATA only)
# ------------------------------------------------------------
if mode == "ADD_NEW_DATA":
    for fld in fields_to_update:
        if fld not in catch_fields:
            if DRY_RUN:
                print(f"[DRY RUN] Would add field {fld} to Catchments_Final")
            else:
                arcpy.management.AddField(catch_fc, fld, "DOUBLE")
                print(f"Added field {fld} to Catchments_Final")

    # refresh field list after creation
    catch_fields = [f.name for f in arcpy.ListFields(catch_fc)]

# ------------------------------------------------------------
# DRY RUN short-circuit for ADD_NEW_DATA
# ------------------------------------------------------------
if DRY_RUN and mode == "ADD_NEW_DATA":
    total_rows = int(arcpy.management.GetCount(catch_fc)[0])

    print("\n--- DRY RUN SUMMARY ---")
    print(f"Fields to be added and populated: {fields_to_update}")
    print(f"Total catchments to receive new values: {total_rows}")
    print("All values will be sourced via IsMainstem logic:")
    print("  IsMainstem = 1 → UPSTREAM HUC values")
    print("  IsMainstem = 0 → LOCAL HUC values")
    print()

    print(
        f"Summary: rows affected = {total_rows}, "
        f"field values changed = {total_rows * len(fields_to_update)}, "
        f"mode = {mode}, DRY_RUN = {DRY_RUN}"
    )

    print("\n=== CELL 3 END (DRY RUN) ===")




# ------------------------------------------------------------
# Guard: ensure fields exist when required
# ------------------------------------------------------------
missing = [f for f in fields_to_update if f not in catch_fields]
if missing and mode != "ADD_NEW_DATA":
    raise RuntimeError(f"Catchments_Final is missing expected fields: {missing}")

# ------------------------------------------------------------
# Build optional where clause
# ------------------------------------------------------------
def build_where(fc, field, values):
    if not values:
        return None
    fld = arcpy.ListFields(fc, field)[0]
    delim = arcpy.AddFieldDelimiters(fc, field)
    if fld.type == "String":
        quoted = ",".join(f"'{v}'" for v in values)
    else:
        quoted = ",".join(str(v) for v in values)
    return f"{delim} IN ({quoted})"

where_clause = build_where(
    catch_fc,
    catch_huc_field,
    target_hucs if mode == "RECALCULATE_FOR_HUC12" else None
)

# ------------------------------------------------------------
# Update Catchments_Final
# ------------------------------------------------------------

if not (DRY_RUN and mode == "ADD_NEW_DATA"):

    comid_field = "comid"
    update_fields = [comid_field, catch_huc_field, is_mainstem_f] + fields_to_update

    from collections import defaultdict

    # DRY_RUN summary counters
    huc_update_counts = defaultdict(int)
    total_updates = 0

    Cursor = arcpy.da.SearchCursor if DRY_RUN else arcpy.da.UpdateCursor

    rows_touched = 0
    field_writes = 0

    with Cursor(catch_fc, update_fields, where_clause) as cur:
        for row in cur:
            comid = row[0]
            huc = str(row[1])
            is_mainstem = row[2]

            src = up_vals if is_mainstem == 1 else local_vals
            if huc not in src:
                continue

            row_changed = False

            if DRY_RUN:
                diffs = []

                for i, fld in enumerate(fields_to_update, start=3):
                    old_val = row[i]
                    new_val = src[huc].get(fld)

                    if old_val != new_val:
                        diffs.append(f"{fld}: {old_val} → {new_val}")
                        field_writes += 1
                        row_changed = True

                if diffs:
                    huc_update_counts[huc] += 1
                    total_updates += 1

            else:
                row = list(row)

                for i, fld in enumerate(fields_to_update, start=3):
                    old_val = row[i]
                    new_val = src[huc].get(fld)

                    if old_val != new_val:
                        row[i] = new_val
                        field_writes += 1
                        row_changed = True

                if row_changed:
                    rows_touched += 1
                    cur.updateRow(row)

    if DRY_RUN:
        print("\n--- DRY RUN SUMMARY (rows that would change) ---")
        for huc in sorted(huc_update_counts):
            print(f"HUC {huc} : {huc_update_counts[huc]} catchments")
        print(f"TOTAL            : {total_updates} catchments\n")

    print(
        f"\nSummary: rows affected = {rows_touched}, "
        f"field values changed = {field_writes}, "
        f"mode = {mode}, DRY_RUN = {DRY_RUN}"
    )




=== CELL 3 START ===
Loaded HUC values: local=6283, upstream=6283
Fields to update: ['Aridity_daymetws', 'DepthToBedrockMean_USGS', 'ElevMedian_daymetws', 'InterAnnPrecipCV_daymetws', 'PercPrecipSnow_daymetws', 'Permeability_permafrost', 'Porosity', 'PotEvapotransEra5_daymetws', 'PrecDaysMean_daymetws', 'PrecMean_daymetws', 'PrecipIntensity_daymetws', 'PrecipSeasInd_daymetws', 'SRadMean_daymetws', 'SWE_April_daymetws', 'SimpPrecIntensity_daymetws', 'SoilWaterContent_WCPF2Mean', 'TempJulyMean_daymetws', 'TempMax_daymetws', 'TempMean_daymetws', 'TempMin_daymetws', 'WinterSumPrecipRatio_daymetws', 'aet_mm_syr', 'ari_ix_sav', 'cly_pc_sav', 'cmi_ix_syr', 'run_mm_syr', 'slt_pc_sav', 'snd_pc_sav']

Summary: rows affected = 78, field values changed = 480, mode = RECALCULATE_FOR_HUC12, DRY_RUN = False
